# 🎙️ Notebook 1: Arabic ASR with OpenAI Whisper

**Project:** Deep Learning Based Arabic Audio Understanding and Retrieval System  
**Task:** Speech-to-Text (ASR)  
**Model:** OpenAI Whisper `large-v3-turbo`  
**Dataset:** Arabic TTS (Common Voice Arabic) — `mayarjao/arabic-tts` on Kaggle  
**Metrics:** WER (Word Error Rate), CER (Character Error Rate)  

**Expected Runtime:** ~3-5 minutes on Kaggle T4 (200 samples)  
**Expected WER:** 15–40% on Arabic speech

In [ ]:
# Cell 1: Install dependencies
!pip install -q openai-whisper jiwer soundfile tqdm
print("✅ Installation complete.")

In [ ]:
# Cell 2: Imports & GPU Check
import whisper
import torch
import time
import os
import glob
import tempfile
import difflib
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
from jiwer import wer, cer
from tqdm.auto import tqdm

print("=" * 60)
print("GPU / SYSTEM CHECK")
print("=" * 60)
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA: {torch.version.cuda}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ NO GPU! Go to Settings → Accelerator → GPU T4")
print("=" * 60)

## 1. Dataset Loading

We load the **Arabic TTS** dataset from Kaggle (`mayarjao/arabic-tts`).  
This dataset contains ~78K Arabic speech samples from Mozilla Common Voice with metadata.

**Important:** Add the dataset via Kaggle sidebar: **Add Data → Search `arabic-tts`**

In [ ]:
# Cell 3: Load Arabic TTS dataset from Kaggle input
DATASET_ROOT = "/kaggle/input/arabic-tts/arabic_tts"
WAV_DIR = os.path.join(DATASET_ROOT, "wavs")
META_PATH = os.path.join(DATASET_ROOT, "metadata.csv")

NUM_SAMPLES = 200  # 200 is enough for solid evaluation

def load_local_arabic_tts(n_samples=NUM_SAMPLES):
    print(f"📁 Loading from: {DATASET_ROOT}")
    
    # Read metadata — columns: col0=text, col1=filename
    df_meta = pd.read_csv(META_PATH, sep="|", header=None, on_bad_lines="skip")
    print(f"   Raw metadata shape: {df_meta.shape}")
    
    df_meta = df_meta.iloc[:, :2]
    df_meta.columns = ["text", "filename"]
    df_meta = df_meta.dropna()
    df_meta = df_meta[df_meta["filename"].astype(str).str.contains(".wav", case=False, na=False)]
    df_meta = df_meta[df_meta["text"].astype(str).str.len() > 2]
    
    print(f"   Valid entries: {len(df_meta)}")
    
    # Build WAV lookup
    wav_lookup = {}
    for wav_path in glob.glob(os.path.join(WAV_DIR, "*.wav")):
        base = os.path.basename(wav_path).replace(".wav", "")
        wav_lookup[base] = wav_path
    
    print(f"   WAV files found: {len(wav_lookup)}")
    
    samples = []
    for _, row in tqdm(df_meta.iterrows(), total=len(df_meta), desc="Matching WAVs"):
        if len(samples) >= n_samples:
            break
        
        fname_raw = str(row["filename"]).strip().replace(".wav", "")
        text = str(row["text"]).strip()
        wav_path = wav_lookup.get(fname_raw)
        
        if wav_path is None:
            continue
        
        try:
            arr, sr = sf.read(wav_path)
            if len(arr) == 0:
                continue
            samples.append({
                "id": fname_raw,
                "audio_array": arr,
                "sampling_rate": sr,
                "reference_text": text,
                "source": "ArabicTTS_local"
            })
        except Exception:
            continue
    
    if len(samples) == 0:
        raise RuntimeError("No valid WAV+text pairs loaded.")
    
    print(f"\n✅ Loaded {len(samples)} samples.")
    return samples

samples = load_local_arabic_tts(NUM_SAMPLES)

print(f"\n{'='*60}")
print(f"📊 DATASET SUMMARY")
print(f"{'='*60}")
print(f"Source       : Arabic TTS (mayarjao/arabic-tts)")
print(f"Total samples: {len(samples)}")
ex = samples[0]
dur = len(ex['audio_array']) / ex['sampling_rate']
print(f"First sample : ID={ex['id']}, Duration={dur:.1f}s")
print(f"Reference    : {ex['reference_text'][:120]}")

## 2. Load Whisper Model

We use **Whisper `large-v3-turbo`** for fast, high-quality Arabic ASR.  
This model provides excellent Arabic accuracy with ~1.5B parameters.

If you get CUDA OOM, change to `"medium"` below.

In [ ]:
# Cell 4: Load Whisper model
MODEL_SIZE = "large-v3-turbo"  # Change to "medium" if OOM

print(f"🔄 Loading Whisper: {MODEL_SIZE} ...")
t0 = time.time()
model = whisper.load_model(MODEL_SIZE)
load_time = time.time() - t0

print(f"✅ Loaded in {load_time:.1f}s")
print(f"   Device: {next(model.parameters()).device}")
print(f"   Parameters: ~{1.55 if 'large' in MODEL_SIZE else 0.8}B (approx)")
print(f"\n📌 Model supports Arabic: 'ar' in decode options.")

## 3. Transcription Loop

For each sample:
1. Save audio array to temporary WAV file
2. Run Whisper with `language="ar"` (forced Arabic)
3. Record inference time
4. Store reference vs prediction

In [ ]:
# Cell 5: Transcription loop
results = []

print("=" * 70)
print(f"🎙️ TRANSCRIBING {len(samples)} SAMPLES")
print("=" * 70)

for idx, sample in enumerate(tqdm(samples, desc="ASR")):
    ref = sample["reference_text"]
    if not ref or len(ref.strip()) < 2:
        continue
    
    try:
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
            sf.write(tmp.name, sample["audio_array"], sample["sampling_rate"])
            tmp_path = tmp.name
        
        t1 = time.time()
        out = model.transcribe(
            tmp_path,
            language="ar",
            fp16=torch.cuda.is_available(),
            verbose=False
        )
        infer_time = time.time() - t1
        pred = out["text"].strip()
        
        os.remove(tmp_path)
        
        results.append({
            "id": sample["id"],
            "source": sample["source"],
            "reference": ref,
            "prediction": pred,
            "duration_sec": round(len(sample["audio_array"]) / sample["sampling_rate"], 2),
            "inference_time_sec": round(infer_time, 2)
        })
    except Exception as e:
        print(f"❌ Error on sample {idx} ({sample['id']}): {e}")
        continue

print(f"\n✅ Done: {len(results)} / {len(samples)} successfully transcribed.")
df = pd.DataFrame(results)
print(f"\nPreview:")
display(df.head())

## 4. Evaluation: WER & CER

We compute:
- **Overall WER/CER** on all samples
- **Per-sample WER/CER** to identify best/worst cases

> ⚠️ Arabic references often have diacritics (ً ٌ ٍ) while Whisper outputs do not. This artificially inflates WER slightly.

In [ ]:
# Cell 6: Compute WER & CER
if len(results) == 0:
    raise ValueError("No results to evaluate!")

refs = [r["reference"] for r in results]
preds = [r["prediction"] for r in results]

overall_wer = wer(refs, preds)
overall_cer = cer(refs, preds)

print("=" * 60)
print("📊 OVERALL METRICS")
print("=" * 60)
print(f"Word Error Rate (WER): {overall_wer:.4f}  →  {overall_wer*100:.2f}%")
print(f"Char Error Rate (CER): {overall_cer:.4f}  →  {overall_cer*100:.2f}%")
print(f"Samples evaluated    : {len(results)}")
print("=" * 60)

# Per-sample metrics
wers, cers = [], []
for r in results:
    try:
        wers.append(wer(r["reference"], r["prediction"]))
    except Exception:
        wers.append(1.0)
    try:
        cers.append(cer(r["reference"], r["prediction"]))
    except Exception:
        cers.append(1.0)

df["wer"] = wers
df["cer"] = cers

# Real-Time Factor
df["rtf"] = df["inference_time_sec"] / df["duration_sec"]

# Normalized WER (strip diacritics for fair comparison)
def strip_diacritics(text):
    return re.sub(r'[\u064B-\u065F\u0670]', '', text)

wers_norm = []
cers_norm = []
for r in results:
    try:
        wers_norm.append(wer(strip_diacritics(r["reference"]), strip_diacritics(r["prediction"])))
    except:
        wers_norm.append(1.0)
    try:
        cers_norm.append(cer(strip_diacritics(r["reference"]), strip_diacritics(r["prediction"])))
    except:
        cers_norm.append(1.0)

df["wer_normalized"] = wers_norm
df["cer_normalized"] = cers_norm

print("\n📈 Per-Sample WER Stats:")
print(df["wer"].describe())

print("\n🏆 Best 3 (lowest WER):")
display(df.nsmallest(3, "wer")[["id","source","wer","cer","inference_time_sec"]])

print("\n💥 Worst 3 (highest WER):")
display(df.nlargest(3, "wer")[["id","source","wer","cer","inference_time_sec"]])

In [ ]:
# Cell 7: Error analysis — word-level diff
def word_diff(ref, pred):
    ref_words = ref.split()
    pred_words = pred.split()
    diff = list(difflib.ndiff(ref_words, pred_words))
    out = []
    for d in diff:
        if d.startswith("- "):
            out.append(f"[-{d[2:]}]")
        elif d.startswith("+ "):
            out.append(f"[+{d[2:]}]")
        elif d.startswith("? "):
            continue
        else:
            out.append(d[2:])
    return " ".join(out)

df_sorted = df.sort_values("wer").reset_index(drop=True)
indices = [0, 1, len(df_sorted)//2, len(df_sorted)-2, len(df_sorted)-1]
examples = df_sorted.iloc[indices]

print("=" * 80)
print("🔍 ERROR ANALYSIS: 5 EXAMPLES (Best → Median → Worst)")
print("=" * 80)

for i, (_, row) in enumerate(examples.iterrows(), 1):
    print(f"\n--- Example {i}/5 | ID: {row['id']} | WER: {row['wer']:.2%} ---")
    print(f"📖 REFERENCE: {row['reference'][:300]}")
    print(f"🤖 PREDICTION: {row['prediction'][:300]}")
    print(f"🔎 DIFF: {word_diff(row['reference'], row['prediction'])[:400]}")
    print("-" * 80)

In [ ]:
# Cell 8: Save results
out_csv = "/kaggle/working/transcripts.csv"
df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"💾 Saved: {out_csv} ({os.path.getsize(out_csv)/1024:.1f} KB)")

print("\n" + "=" * 60)
print("📋 RESULTS TABLE FOR REPORT")
print("=" * 60)
print("| Metric | Value |")
print("|--------|-------|")
print(f"| Model | Whisper {MODEL_SIZE} |")
print(f"| Dataset | Arabic TTS (Common Voice Arabic) |")
print(f"| Samples | {len(df)} |")
print(f"| WER | {overall_wer*100:.2f}% |")
print(f"| CER | {overall_cer*100:.2f}% |")
print(f"| Mean Inference Time | {df['inference_time_sec'].mean():.2f}s |")
print(f"| GPU | Tesla T4 |")
print("=" * 60)

In [ ]:
# Cell 9: Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: WER distribution
ax1 = axes[0]
sorted_wer = df.sort_values("wer")["wer"].values * 100
colors = ['#2ecc71' if x < 20 else '#f39c12' if x < 50 else '#e74c3c' for x in sorted_wer]
ax1.bar(range(len(sorted_wer)), sorted_wer, color=colors, alpha=0.8, edgecolor='black', linewidth=0.3)
ax1.axhline(y=overall_wer*100, color='navy', linestyle='--', linewidth=2, label=f'Overall WER = {overall_wer*100:.1f}%')
ax1.set_xlabel("Sample Index (sorted by WER)", fontsize=11)
ax1.set_ylabel("Word Error Rate (%)", fontsize=11)
ax1.set_title("Per-Sample WER Distribution", fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Plot 2: Inference time vs Duration
ax2 = axes[1]
scatter = ax2.scatter(df["duration_sec"], df["inference_time_sec"],
                     c=df["wer"], cmap='RdYlGn_r', alpha=0.7, edgecolors='black', linewidth=0.5)
ax2.set_xlabel("Audio Duration (seconds)", fontsize=11)
ax2.set_ylabel("Inference Time (seconds)", fontsize=11)
ax2.set_title("Inference Time vs Duration (colored by WER)", fontsize=13, fontweight='bold')
cbar = plt.colorbar(scatter, ax=ax2)
cbar.set_label("WER", rotation=270, labelpad=15)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/wer_analysis.png", dpi=200, bbox_inches="tight")
plt.show()
print("💾 Plot saved: /kaggle/working/wer_analysis.png")

## Related Work

1. **Whisper:** Radford, A., et al. (2023). "Robust Speech Recognition via Large-Scale Weak Supervision." *ICML 2023.* OpenAI Whisper was trained on 680,000 hours of multilingual audio and achieves state-of-the-art zero-shot ASR.

2. **Common Voice:** Ardila, R., et al. (2020). "Common Voice: A Massively-Multilingual Speech Corpus." *LREC 2020.* Crowdsourced multilingual dataset including Arabic.

### Known Limitations
- **Diacritics:** Whisper outputs undiacritized Arabic. References with diacritics inflate WER.
- **Dialects:** Performance degrades on Egyptian, Levantine, Gulf, and Maghrebi dialects.
- **Noise:** Background noise increases error rates.

## Results Summary

| Item | Value |
|------|-------|
| Model | OpenAI Whisper `large-v3-turbo` |
| Dataset | Arabic TTS (Common Voice Arabic) |
| Samples | *(fill after running)* |
| **Overall WER** | ***(fill after running)*** |
| **Overall CER** | ***(fill after running)*** |
| GPU Used | Tesla T4 |